### Raster Model
This notebook contains the workflow for downscaling national capital and labor data. First, it uses the RF model to predict intensities at the pixel scale. Then it combines intensities with rasterized production data to get initial estimates of capital and labor in each pixel. Then it runs a re-scaling algorithim to ensure estimates align with sub-national (where available) and national totals. The output is a set of raster datasets contained estimates of capital (million USD) and labor (jobs) in each pixel. 

In [1]:
import pandas as pd
from pathlib import Path
import numpy as np
import joblib
import rasterio
import gc

In [2]:
##### SET-UP

### Set directories 
# Get the current working directory
cd = Path.cwd().parent 

# Set directories for saving
RESULTS_DIR = Path(f"{cd}/Results/Raster_models")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

### Import data 
# RF models 
capital_rf = joblib.load(f'{cd}/Results/RF_models/capital_rf_final/model.joblib')
capital_qrf = joblib.load(f'{cd}/Results/RF_models/capital_qrf_final/model.joblib')

labor_rf = joblib.load(f'{cd}/Results/RF_models/labor_rf_final/model.joblib')
labor_qrf = joblib.load(f'{cd}/Results/RF_models/labor_qrf_final/model.joblib')

# Predictor data 
predictors = pd.read_parquet(f'{cd}/Data/raster_predictor_matrix_final.parquet')

# Production data 
production_path = f'{cd}/Data/total_production_tonnes_2020.tif'

# Sub-national data (for re-scaling)
capital_sub = pd.read_csv(f'{cd}/Data/subnational_capital_stock.csv')
capital_sub_crosswalk = pd.read_parquet(f'{cd}/Data/pixel_sub_capital_crosswalk.parquet')

labor_sub = pd.read_csv(f'{cd}/Data/subnational_labor.csv')
labor_sub_crosswalk = pd.read_parquet(f'{cd}/Data/pixel_sub_labor_crosswalk.parquet')

# National data (for re-scaling)
national_intensities = pd.read_csv(f'{cd}/Data/national_data.csv')
country_crosswalk = pd.read_parquet(f'{cd}/Data/pixel_country_crosswalk.parquet')

#### Part 1: Predict intensities

In [3]:
##### PREDICTOR COLUMNS

capital_cols = ['rtv_log_average_travel_time_port',
       'rtv_log_crop_intensity',
       'rtv_log_USD_production_per_million_HA',
       'rtv_log_tonnes_production_per_million_HA',
       'rtv_log_pop_density_people_per_100_km2',
       'rtv_log_cattle_density_per_100_km2',
       'rtv_log_sheep_density_per_100_km2',
       'rtv_log_livestock_density_LU_per_100_km2',
       'rtv_log_cereals_share_base_100_production_tonnes',
       'rtv_log_fruits_share_base_100_production_tonnes',
       'rtv_log_roots_tubers_share_base_100_production_tonnes',
       'rtv_log_vegetables_share_base_100_production_tonnes',
       'rtv_log_ruminants_share_base_100_production_tonnes',
       'rtv_log_share_base_100_large_field',
       'rtv_log_share_base_100_with_nightlights']

labor_cols = ['rtv_log_average_travel_time_port',
       'rtv_log_crop_intensity',
       'rtv_log_pop_density_people_per_100_km2',
       'rtv_log_cattle_density_per_100_km2',
       'rtv_log_livestock_density_LU_per_100_km2',
       'rtv_log_fruits_share_base_100_production_tonnes',
       'rtv_log_roots_tubers_share_base_100_production_tonnes',
       'rtv_log_rest_of_crops_share_base_100_production_tonnes',
       'rtv_log_sugar_crops_share_base_100_production_tonnes',
       'rtv_log_ruminants_share_base_100_production_tonnes',
       'rtv_log_pct_base_100_GDP_ag', 'rtv_log_share_base_100_large_field',
       'rtv_log_pct_base_100_cropland_irrigated']

In [4]:
##### PREDICT INTENSITIES 

# reduce parallelism to limit memory multiplication 
capital_qrf.n_jobs = 4
labor_qrf.n_jobs = 4
capital_rf.n_jobs = 4
labor_rf.n_jobs = 4

chunk_size = 50_000  
n = len(predictors)

# set output columns
out_cols = [
    'rtv_log_capital_intensity_USD_per_million_tonne',
    'rtv_log_labor_intensity_jobs_per_million_tonne',
    'rtv_log_capital_intensity_p10',
    'rtv_log_capital_intensity_p90',
    'rtv_log_labor_intensity_p10',
    'rtv_log_labor_intensity_p90',
]

for c in out_cols:
    predictors[c] = np.nan

# predict!
for start in range(0, n, chunk_size):
    end = min(start + chunk_size, n)
    idx = predictors.index[start:end]

    X_cap_chunk = predictors.loc[idx, capital_cols].astype('float32')
    X_lab_chunk = predictors.loc[idx, labor_cols].astype('float32')

    predictors.loc[idx, 'rtv_log_capital_intensity_USD_per_million_tonne'] = capital_rf.predict(X_cap_chunk)
    predictors.loc[idx, 'rtv_log_labor_intensity_jobs_per_million_tonne'] = labor_rf.predict(X_lab_chunk)

    cap_q = capital_qrf.predict(X_cap_chunk, quantiles=[0.1, 0.9])
    predictors.loc[idx, 'rtv_log_capital_intensity_p10'] = cap_q[:, 0]
    predictors.loc[idx, 'rtv_log_capital_intensity_p90'] = cap_q[:, 1]

    lab_q = labor_qrf.predict(X_lab_chunk, quantiles=[0.1, 0.9])
    predictors.loc[idx, 'rtv_log_labor_intensity_p10'] = lab_q[:, 0]
    predictors.loc[idx, 'rtv_log_labor_intensity_p90'] = lab_q[:, 1]

    del X_cap_chunk, X_lab_chunk, cap_q, lab_q
    gc.collect()

    print(f"done {end}/{n}")

# Filter 
cols_to_keep = ['x', 'y',
    'rtv_log_capital_intensity_USD_per_million_tonne',
    'rtv_log_labor_intensity_jobs_per_million_tonne',
    'rtv_log_capital_intensity_p10',
    'rtv_log_capital_intensity_p90',
    'rtv_log_labor_intensity_p10',
    'rtv_log_labor_intensity_p90',
]
raster_predictions = predictors[cols_to_keep]

done 50000/1948435
done 100000/1948435
done 150000/1948435
done 200000/1948435
done 250000/1948435
done 300000/1948435
done 350000/1948435
done 400000/1948435
done 450000/1948435
done 500000/1948435
done 550000/1948435
done 600000/1948435
done 650000/1948435
done 700000/1948435
done 750000/1948435
done 800000/1948435
done 850000/1948435
done 900000/1948435
done 950000/1948435
done 1000000/1948435
done 1050000/1948435
done 1100000/1948435
done 1150000/1948435
done 1200000/1948435
done 1250000/1948435
done 1300000/1948435
done 1350000/1948435
done 1400000/1948435
done 1450000/1948435
done 1500000/1948435
done 1550000/1948435
done 1600000/1948435
done 1650000/1948435
done 1700000/1948435
done 1750000/1948435
done 1800000/1948435
done 1850000/1948435
done 1900000/1948435
done 1948435/1948435


In [5]:
##### CONVERT UNITS

# add back country intensities 
raster_predictions = raster_predictions.merge(predictors[['x', 'y', 'log_country_capital_intensity_USD_per_million_tonne', 'log_country_labor_intensity_jobs_per_million_tonne']], on=['x', 'y'], how='outer')

# convert pixel intensities from relative to absolute 
raster_predictions['capital_intensity_USD_per_million_tonne'] = np.expm1(raster_predictions['rtv_log_capital_intensity_USD_per_million_tonne'] + raster_predictions['log_country_capital_intensity_USD_per_million_tonne'])
raster_predictions['capital_intensity_p10'] = np.expm1(raster_predictions['rtv_log_capital_intensity_p10'] + raster_predictions['log_country_capital_intensity_USD_per_million_tonne'])
raster_predictions['capital_intensity_p90'] = np.expm1(raster_predictions['rtv_log_capital_intensity_p90'] + raster_predictions['log_country_capital_intensity_USD_per_million_tonne'])

raster_predictions['labor_intensity_jobs_per_million_tonne'] = np.expm1(raster_predictions['rtv_log_labor_intensity_jobs_per_million_tonne'] + raster_predictions['log_country_labor_intensity_jobs_per_million_tonne'])
raster_predictions['labor_intensity_p10'] = np.expm1(raster_predictions['rtv_log_labor_intensity_p10'] + raster_predictions['log_country_labor_intensity_jobs_per_million_tonne'])
raster_predictions['labor_intensity_p90'] = np.expm1(raster_predictions['rtv_log_labor_intensity_p90'] + raster_predictions['log_country_labor_intensity_jobs_per_million_tonne'])

# save intermediate output
output_dir = RESULTS_DIR / "intermediate_outputs"
output_dir.mkdir(parents=True, exist_ok=True)

raster_predictions.to_parquet(
    output_dir / "intensity_predictions.parquet",
    index=False
)

#### Part 2: Initial capital and labor estimates

In [6]:
##### PREP PRODUCTION DATA 

# convert production tiff into df with x/y coords 
# both prediction and production raster in EPSG:4326, 5 arcmin - so x/y match 
with rasterio.open(production_path) as src:
    arr = src.read(1).astype(np.float32)
    nodata = src.nodata
    if nodata is not None:
        arr[arr == nodata] = np.nan

    height, width = arr.shape
    transform = src.transform
    rows, cols = np.meshgrid(np.arange(height), np.arange(width), indexing="ij")
    xs, ys = rasterio.transform.xy(transform, rows.ravel(), cols.ravel())
    xs = np.array(xs)
    ys = np.array(ys)

production_df = pd.DataFrame({
    "x": xs,
    "y": ys,
    "total_production_tonnes_2020": arr.ravel()
})

In [7]:
##### CONVERT INTENSITIES TO CAPITAL AND LABOR

# Merge intensities and production
raster_predictions = raster_predictions.merge(production_df, on=['x', 'y'], how='outer')

# intensity to total conversion
# total = intensity_per_million_tonne * (production_tonnes / 1e6)
production_million_tonnes = raster_predictions['total_production_tonnes_2020'] / 1e6

conversion_map = {
    'capital_intensity_USD_per_million_tonne': 'capital_USD',
    'capital_intensity_p10':                    'capital_USD_p10',
    'capital_intensity_p90':                    'capital_USD_p90',
    'labor_intensity_jobs_per_million_tonne':   'jobs',
    'labor_intensity_p10':                       'jobs_p10',
    'labor_intensity_p90':                       'jobs_p90',
}

for intensity_col, total_col in conversion_map.items():
    raster_predictions[total_col] = raster_predictions[intensity_col] * production_million_tonnes

# Filter columns 
col_to_keep = ['x', 'y', 'total_production_tonnes_2020'] + list(conversion_map.values())
capital_labor_predictions = raster_predictions[col_to_keep]

# save intermediate output
capital_labor_predictions.to_parquet(RESULTS_DIR / "intermediate_outputs/capital_labor_predictions.parquet", index=False)

In [8]:
##### SAVE INTERMEDIATE RASTERS

# define function to convert a df column back into a raster, using a reference grid
def save_column_as_raster(df, value_col, reference_raster_path, output_path):

    # use production raster as reference grid
    with rasterio.open(reference_raster_path) as ref:
        transform = ref.transform
        crs = ref.crs
        height = ref.height
        width = ref.width

    # pivot into 2D grid: rows = y, cols = x
    pivot = df.pivot(index='y', columns='x', values=value_col)
    pivot = pivot.sort_index(ascending=False)
    pivot = pivot.reindex(columns=sorted(pivot.columns))

    arr = pivot.values.astype(np.float32)

    if arr.shape != (height, width):
        raise ValueError(
            f"Shape mismatch for {value_col}: got {arr.shape}, expected {(height, width)}"
        )

    with rasterio.open(
        output_path, 'w',
        driver='GTiff',
        height=height,
        width=width,
        count=1,
        dtype='float32',
        crs=crs,
        transform=transform,
        nodata=np.nan
    ) as dst:
        dst.write(arr, 1)

# run for all columns 
output_cols = list(conversion_map.values())

for col in output_cols:
    out_path = RESULTS_DIR / f"intermediate_outputs/pre_scaled_{col}.tif" 
    save_column_as_raster(raster_predictions, col, production_path, out_path)
    print(f"Saved {col} -> {out_path}")

Saved capital_USD -> /Users/carinamanitius/Documents/GitHub/Gridded_Agricultural_Capital_Labor/Results/Raster_models/intermediate_outputs/pre_scaled_capital_USD.tif
Saved capital_USD_p10 -> /Users/carinamanitius/Documents/GitHub/Gridded_Agricultural_Capital_Labor/Results/Raster_models/intermediate_outputs/pre_scaled_capital_USD_p10.tif
Saved capital_USD_p90 -> /Users/carinamanitius/Documents/GitHub/Gridded_Agricultural_Capital_Labor/Results/Raster_models/intermediate_outputs/pre_scaled_capital_USD_p90.tif
Saved jobs -> /Users/carinamanitius/Documents/GitHub/Gridded_Agricultural_Capital_Labor/Results/Raster_models/intermediate_outputs/pre_scaled_jobs.tif
Saved jobs_p10 -> /Users/carinamanitius/Documents/GitHub/Gridded_Agricultural_Capital_Labor/Results/Raster_models/intermediate_outputs/pre_scaled_jobs_p10.tif
Saved jobs_p90 -> /Users/carinamanitius/Documents/GitHub/Gridded_Agricultural_Capital_Labor/Results/Raster_models/intermediate_outputs/pre_scaled_jobs_p90.tif


#### Part 3: rescaling to sub-national and national totals

In [9]:
##### Define re-scaling function

def rescale_predictions(df, group_col, pred_cols, actual_df, actual_id_col, actual_col,
                          scale_prefix, out_prefix, missing_scale=1, save_path=None,
                          suffixes=("", "_p10", "_p90")):

    scale_cols = [f"{scale_prefix}{s}" for s in suffixes]
    out_cols   = [f"{out_prefix}{s}" for s in suffixes]

    # Step 1: sum predictions by group
    agg = df.groupby(group_col)[pred_cols].sum().reset_index()

    # Step 2: compute scale factor = actual / predicted
    agg = agg.merge(actual_df[[actual_id_col, actual_col]],
                     left_on=group_col, right_on=actual_id_col, how='left')
    for pred_col, scale_col in zip(pred_cols, scale_cols):
        agg[scale_col] = agg[actual_col] / agg[pred_col]
    agg[scale_cols] = agg[scale_cols].replace([np.inf, -np.inf], np.nan)

    if save_path is not None:
        agg.to_csv(save_path, index=False)

    # Step 3: apply scale factors to pixel level 
    agg_indexed = agg.set_index(group_col)
    for scale_col in scale_cols:
        df[scale_col] = df[group_col].map(agg_indexed[scale_col])
        if missing_scale is not None:
            df[scale_col] = df[scale_col].fillna(missing_scale)

    for pred_col, scale_col, out_col in zip(pred_cols, scale_cols, out_cols):
        df[out_col] = df[pred_col] * df[scale_col]

    return df, agg


In [10]:
##### RE-SCALE TO SUB-NATIONAL (where it exists)

capital_labor_predictions = pd.read_parquet(RESULTS_DIR / "intermediate_outputs/capital_labor_predictions.parquet")

### Step 1: crosswalk of which sub-national region each pixel is in
capital_sub_crosswalk = capital_sub_crosswalk.rename(columns={'PROJ_ID': 'CAPITAL_PROJ_ID'})
labor_sub_crosswalk = labor_sub_crosswalk.rename(columns={'PROJ_ID': 'LABOR_PROJ_ID'})

capital_labor_predictions = capital_labor_predictions.merge(capital_sub_crosswalk, on=['x', 'y'], how='left')
capital_labor_predictions = capital_labor_predictions.merge(labor_sub_crosswalk, on=['x', 'y'], how='left')

### Step 2: prep known sub-national totals
# For capital, drop sub-national regions in countries where a proxy was used to aprx sub-national totals 
countries_to_drop = ['BRA', 'EGY', 'GHA', 'IND', 'TUR', 'ZAF', 'TZA', 'ARG']
capital_sub = capital_sub[~capital_sub["ISO3"].isin(countries_to_drop)]

# no duplicates
capital_sub = capital_sub[~capital_sub.duplicated(subset=['PROJ_ID'], keep=False)]
labor_sub = labor_sub[~labor_sub.duplicated(subset=['PROJ_ID'], keep=False)]

capital_sub = capital_sub.rename(columns={'ag_capital_stock_USD_nominal': 'sub_national_total_capital_USD'})
labor_sub = labor_sub.rename(columns={'ag_jobs': 'sub_national_total_jobs'})

### Step 3: compute and apply sub-national rescaling
# missing_scale=1: pixels outside any sub-national region keep their original predicted value
capital_labor_predictions, capital_sub_agg = rescale_predictions(
    capital_labor_predictions, group_col='CAPITAL_PROJ_ID',
    pred_cols=['capital_USD', 'capital_USD_p10', 'capital_USD_p90'],
    actual_df=capital_sub, actual_id_col='PROJ_ID', actual_col='sub_national_total_capital_USD',
    scale_prefix='capital_sub_scale', out_prefix='sub_scaled_capital_USD', missing_scale=1,
    save_path=RESULTS_DIR / "intermediate_outputs/subnational_scaling_factors_capital.csv",
)

labor_labor_predictions, labor_sub_agg = rescale_predictions(
    capital_labor_predictions, group_col='LABOR_PROJ_ID',
    pred_cols=['jobs', 'jobs_p10', 'jobs_p90'],
    actual_df=labor_sub, actual_id_col='PROJ_ID', actual_col='sub_national_total_jobs',
    scale_prefix='labor_sub_scale', out_prefix='sub_scaled_jobs', missing_scale=1,
    save_path=RESULTS_DIR / "intermediate_outputs/subnational_scaling_factors_labor.csv",
)
capital_labor_predictions = labor_labor_predictions  

In [11]:
##### RE-SCALE TO NATIONAL (drop where known total doesn't exist)

### Step 1: crosswalk of which country each pixel is in
country_crosswalk = country_crosswalk.rename(columns={'GID_0': 'ISO3'})
capital_labor_predictions = capital_labor_predictions.merge(country_crosswalk, on=['x', 'y'], how='left')

### Step 2: prep known national totals
national_intensities['national_total_capital_USD'] = national_intensities['ag_capital_stock_mil_USD_nominal'] * 1e6
capital_national = national_intensities[['ISO3', 'national_total_capital_USD']]

national_intensities['national_total_jobs'] = national_intensities['ag_labor_thousands_2020'] * 1e3
labor_national = national_intensities[['ISO3', 'national_total_jobs']]

### Step 3: compute and apply national rescaling
# missing_scale=None: pixels in a country with no known national total are dropped
capital_labor_predictions, capital_nat_agg = rescale_predictions(
    capital_labor_predictions, group_col='ISO3',
    pred_cols=['sub_scaled_capital_USD', 'sub_scaled_capital_USD_p10', 'sub_scaled_capital_USD_p90'],
    actual_df=capital_national, actual_id_col='ISO3', actual_col='national_total_capital_USD',
    scale_prefix='capital_nat_scale', out_prefix='rescaled_capital_USD', missing_scale=None,
    save_path=RESULTS_DIR / "intermediate_outputs/national_scaling_factors_capital.csv",
)

capital_labor_predictions, labor_nat_agg = rescale_predictions(
    capital_labor_predictions, group_col='ISO3',
    pred_cols=['sub_scaled_jobs', 'sub_scaled_jobs_p10', 'sub_scaled_jobs_p90'],
    actual_df=labor_national, actual_id_col='ISO3', actual_col='national_total_jobs',
    scale_prefix='labor_nat_scale', out_prefix='rescaled_jobs', missing_scale=None,
    save_path=RESULTS_DIR / "intermediate_outputs/national_scaling_factors_labor.csv",
)

In [12]:
##### SANITY CHECKS

# Check that total of known data matches total of raster data
cap_err_per = (capital_labor_predictions['rescaled_capital_USD'].sum() / capital_national['national_total_capital_USD'].sum()) * 100
lab_err_per = (capital_labor_predictions['rescaled_jobs'].sum() / labor_national['national_total_jobs'].sum()) * 100

print("Percentage difference between sum of pixels and sum of countries:")
print(f"Capital: {cap_err_per:.2f}")
print(f"Labor: {lab_err_per:.2f}")

Percentage difference between sum of pixels and sum of countries:
Capital: 99.99
Labor: 99.99


In [13]:
##### SAVE FINAL OUTPUTS 

# save intermediate output
capital_labor_predictions.to_parquet(RESULTS_DIR / "intermediate_outputs/capital_labor_predictions_rescaled.parquet", index=False)

# final clean 
col_to_keep = ['x', 'y',
               'rescaled_capital_USD', 'rescaled_capital_USD_p10', 'rescaled_capital_USD_p90',
               'rescaled_jobs', 'rescaled_jobs_p10', 'rescaled_jobs_p90']
capital_labor_final = capital_labor_predictions[col_to_keep]

capital_labor_final.to_parquet(RESULTS_DIR / "capital_labor_final.parquet", index=False)

# save rasters
output_cols = [
    'rescaled_capital_USD',
    'rescaled_capital_USD_p10',
    'rescaled_capital_USD_p90',
    'rescaled_jobs',
    'rescaled_jobs_p10',
    'rescaled_jobs_p90'
]

for col in output_cols:
    out_path = RESULTS_DIR / f"{col}.tif" 
    save_column_as_raster(capital_labor_final, col, production_path, out_path)
    print(f"Saved {col} -> {out_path}")

Saved rescaled_capital_USD -> /Users/carinamanitius/Documents/GitHub/Gridded_Agricultural_Capital_Labor/Results/Raster_models/rescaled_capital_USD.tif
Saved rescaled_capital_USD_p10 -> /Users/carinamanitius/Documents/GitHub/Gridded_Agricultural_Capital_Labor/Results/Raster_models/rescaled_capital_USD_p10.tif
Saved rescaled_capital_USD_p90 -> /Users/carinamanitius/Documents/GitHub/Gridded_Agricultural_Capital_Labor/Results/Raster_models/rescaled_capital_USD_p90.tif
Saved rescaled_jobs -> /Users/carinamanitius/Documents/GitHub/Gridded_Agricultural_Capital_Labor/Results/Raster_models/rescaled_jobs.tif
Saved rescaled_jobs_p10 -> /Users/carinamanitius/Documents/GitHub/Gridded_Agricultural_Capital_Labor/Results/Raster_models/rescaled_jobs_p10.tif
Saved rescaled_jobs_p90 -> /Users/carinamanitius/Documents/GitHub/Gridded_Agricultural_Capital_Labor/Results/Raster_models/rescaled_jobs_p90.tif
